# 🚀 ADAPT ATC — GRPO Training | A100 GPU (bf16)

**High-throughput training on A100 — 2× batch, 4 generations, bf16**

| Setting | T4 notebook | **A100 notebook** |
|---------|-------------|-------------------|
| dtype | fp16 | **bf16** |
| Batch | 1 × accum 4 | **4 × accum 2** |
| Generations | 2 | **4** |
| Completion len | 256 | **512** |
| Episodes | 150 | **300** |
| Est. time | ~2 h | **~2–3 h** |

> **One-click**: Runtime → Change runtime type → A100 GPU → Run All

## Why bf16 on A100?
A100 has native bf16 Tensor Core support. bf16 gives identical dynamic range to fp32
(8 exponent bits vs 5 in fp16) while matching fp16 memory footprint.
**Never mix bf16 and fp16 in the same training run** — the GRPOConfig flags
`bf16=True` and `fp16=False` here are explicit, not auto-detected.

## What trains
Three agents share one LLM (system-prompt-differentiated):
- **AMAN** – arrival sequencing (wake turbulence, emergency priority)
- **DMAN** – departure sequencing (ATFM deadlines, emergency handling)  
- **ADAPT** – domain transfer (maps ICU/port signals → ATC parameters)


## Step 1 — Install dependencies

In [ ]:
import subprocess, sys, os

def pip(*args):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], check=False)

pip("unsloth")
pip("trl>=0.15", "datasets", "bitsandbytes", "accelerate", "peft", "matplotlib")

# Optional: flash-attn for A100 (gives ~15% speed boost, not required)
# pip("flash-attn", "--no-build-isolation")

os.environ["TOKENIZERS_PARALLELISM"] = "false"
print("✅ Dependencies installed")

## Step 2 — GPU check + dtype assertion

In [ ]:
import torch

assert torch.cuda.is_available(), "❌ No GPU! Runtime → Change runtime type → A100 GPU"

gpu_name = torch.cuda.get_device_name(0)
vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU  : {gpu_name}")
print(f"VRAM : {vram_gb:.1f} GB")

# A100/H100 = Ampere/Hopper → bf16 Tensor Core support
IS_BF16 = torch.cuda.is_bf16_supported()
if not IS_BF16:
    print("⚠️  bf16 not supported on this GPU!")
    print("   This notebook requires A100/H100/RTX-3090+ for bf16.")
    print("   For T4 (fp16), use train_t4.ipynb")
    raise RuntimeError("A100 required for this notebook")

print(f"dtype: bfloat16 ✅ (Ampere Tensor Core support confirmed)")
print(f"VRAM : {vram_gb:.0f} GB available")

## Step 3 — Repository setup

**Option A** (recommended): Upload `adapt-atc-final.zip` via Colab Files panel.

**Option B**: Mount Google Drive.

**Option C**: Clone from GitHub.

In [ ]:
import os, sys, shutil

REPO_PATH = "/content/adapt-atc-final"

# ── Option A: unzip uploaded file ────────────────────────────────────────────
# !unzip -q /content/adapt-atc-final.zip -d /content/

# ── Option B: Google Drive ────────────────────────────────────────────────────
# from google.colab import drive
# drive.mount('/content/drive')
# DRIVE = "/content/drive/MyDrive/adapt-atc-final"
# if os.path.exists(DRIVE) and not os.path.exists(REPO_PATH):
#     shutil.copytree(DRIVE, REPO_PATH)

# ── Option C: GitHub clone ────────────────────────────────────────────────────
# if not os.path.exists(REPO_PATH):
#     !git clone https://github.com/YOUR_USERNAME/adapt-atc-final.git /content/adapt-atc-final

# ── Verify ───────────────────────────────────────────────────────────────────
if not os.path.exists(REPO_PATH):
    raise FileNotFoundError(
        f"Repo not found at {REPO_PATH}.\n"
        "Upload adapt-atc-final.zip → Colab Files → uncomment Option A above."
    )

sys.path.insert(0, REPO_PATH)
os.chdir(REPO_PATH)
print(f"✅ Repo ready: {REPO_PATH}")
print(f"   Files: {sorted(os.listdir(REPO_PATH))[:10]}")

## Step 4 — Import repo modules

In [ ]:
try:
    from tasks import task_catalog
    from training.reward_functions import aman_reward_fn, dman_reward_fn, adapt_reward_fn
    from training.dataset import build_episode_dataset, AMAN_SYSTEM, DMAN_SYSTEM
    from multi_agent.models import AgentRole
    print("✅ All imports OK")
    catalog = task_catalog()
    print(f"   Tasks: {list(catalog.keys())}")
except Exception as e:
    print(f"❌ Import error: {e}")
    print("Ensure REPO_PATH in Step 3 is correct and all repo files are present.")
    raise

## Step 5 — Load model (bf16, 4-bit QLoRA)

In [ ]:
from unsloth import FastLanguageModel, PatchFastRL

# CRITICAL: Patch BEFORE loading the model
try:
    PatchFastRL("GRPO", FastLanguageModel)
    print("✅ PatchFastRL applied")
except Exception as e:
    print(f"ℹ️  PatchFastRL: {e}  (newer Unsloth may not need this — continuing)")

# A100 settings: longer context + bf16
# 40 GB VRAM: 4-bit 1.5B = ~900 MB | seq 4096 batch 4 = ~12 GB | LoRA = ~200 MB
MODEL_NAME  = "unsloth/Qwen2.5-1.5B-Instruct"
MAX_SEQ_LEN = 4096   # A100 can handle longer context

print(f"Loading {MODEL_NAME} in 4-bit bf16 (A100 optimized)...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,
    dtype=None,           # auto: bf16 confirmed above via IS_BF16 check
    trust_remote_code=True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token    = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

total = sum(p.numel() for p in model.parameters())
dtype = next(model.parameters()).dtype
print(f"✅ Loaded: {total/1e6:.0f} M params | dtype={dtype}")

## Step 6 — Apply LoRA (rank=16 for A100)

In [ ]:
LORA_RANK = 16  # A100 has headroom for rank=16 (~0.8% trainable params)

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=LORA_RANK * 2,
    lora_dropout=0.0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"✅ LoRA rank={LORA_RANK}  |  {trainable:,} / {total:,} trainable ({100*trainable/total:.2f}%)")

## Step 7 — Build training dataset (300 episodes)

In [ ]:
from datasets import Dataset
from collections import Counter
import time

N_EPISODES = 300  # A100: 300 episodes, ~2-3 hours

print(f"Building {N_EPISODES}-episode dataset ...")
t0 = time.time()

dataset_raw = build_episode_dataset(
    n_episodes=N_EPISODES,
    seed=42,
    include_adapt=True,
    domain_episode_ratio=0.65,
    long_horizon_ratio=0.0,
)
dataset = Dataset.from_list(dataset_raw)

role_counts = Counter(s["agent_role"] for s in dataset_raw)
print(f"✅ {len(dataset)} samples in {time.time()-t0:.1f}s")
for role, cnt in sorted(role_counts.items()):
    print(f"   {role:8s}: {cnt} samples")

## Step 8 — Reward function dispatcher

In [ ]:
_DISPATCH = {
    "AMAN":  aman_reward_fn,
    "DMAN":  dman_reward_fn,
    "ADAPT": adapt_reward_fn,
}


def _partial_credit(text: str, role: str) -> float:
    """Variable partial credit when JSON parsing fully fails."""
    s = 0.05
    if "{" in text: s += 0.03
    if role == "AMAN"  and "arrival_slots"   in text: s += 0.06
    if role == "DMAN"  and "departure_slots" in text: s += 0.06
    if role == "ADAPT" and "entity_wake_map" in text: s += 0.06
    if '"flight_id"'  in text: s += 0.04
    if "rationale"    in text: s += 0.02
    return min(s, 0.22)


def reward_fn(completions, **kwargs):
    """Multi-agent ATC reward dispatcher.
    Returns: List[float] guaranteed — no tensors, no NaN.
    """
    n     = len(completions)
    roles = kwargs.get("agent_role", ["AMAN"] * n)
    if not isinstance(roles, list):
        roles = [roles] * n
    while len(roles) < n:
        roles.append(roles[-1] if roles else "AMAN")

    rewards = []
    for i, (comp, role) in enumerate(zip(completions, roles)):
        if isinstance(comp, list):
            comp = comp[-1].get("content", "") if comp else ""
        comp = str(comp)

        fn = _DISPATCH.get(str(role), aman_reward_fn)

        sample_kw = {}
        for k, v in kwargs.items():
            if k == "agent_role":
                continue
            if isinstance(v, list):
                sample_kw[k] = [v[i] if i < len(v) else (v[-1] if v else "")]
            else:
                sample_kw[k] = [v]

        try:
            r      = fn([comp], **sample_kw)
            reward = float(r[0]) if r else _partial_credit(comp, str(role))
            reward = max(0.0, min(1.0, reward))
            if reward != reward:
                reward = _partial_credit(comp, str(role))
        except Exception:
            reward = _partial_credit(comp, str(role))

        rewards.append(float(reward))

    return rewards


test_r = reward_fn(
    ["{", "{\"arrival_slots\":[{\"flight_id\":\"AI101\",\"runway\":\"28R\",\"assigned_minute\":10,\"hold_minutes\":0}],\"rationale\":\"emergency first\"}"],
    task_id=["delhi_monsoon_recovery_easy", "delhi_monsoon_recovery_easy"],
    agent_role=["AMAN", "AMAN"],
)
assert all(isinstance(r, float) for r in test_r), "Reward must be float!"
print(f"✅ Reward fn OK | partial={test_r[0]:.3f}  full-parse={test_r[1]:.3f}")

## Step 9 — Configure GRPO (A100 optimized)

**A100 key settings vs T4:**
- `bf16=True, fp16=False` — A100 Tensor Core bf16 (larger dynamic range than fp16)
- `num_generations=4` — stable GRPO advantage (N≥4 avoids degenerate std)
- `per_device_train_batch_size=4` — 4× T4 throughput
- `max_completion_length=512` — longer outputs = richer JSON + rationale

In [ ]:
from trl import GRPOConfig, GRPOTrainer

OUTPUT_DIR = "./adapt-atc-a100-out"

training_args = GRPOConfig(
    # ── Generation ────────────────────────────────────────────
    num_generations=4,           # A100: 4 generations = stable GRPO variance
    max_completion_length=512,   # longer outputs = richer plans
    max_prompt_length=3072,      # 3072 + 512 = 3584 < MAX_SEQ_LEN=4096

    # ── Optimisation ──────────────────────────────────────────
    learning_rate=5e-6,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,   # effective batch = 8
    num_train_epochs=3,
    warmup_ratio=0.10,
    max_grad_norm=1.0,

    # ── Precision — EXPLICIT for A100 (never rely on auto-detect) ──
    bf16=True,     # A100 Ampere = bf16 Tensor Cores
    fp16=False,    # fp16 off — never mix with bf16

    # ── Memory ────────────────────────────────────────────────
    optim="adamw_8bit",
    gradient_checkpointing=True,

    # ── KL regularisation ─────────────────────────────────────
    beta=0.0,   # 0 = no KL; avoids ref_per_token_logps=None with PEFT

    # ── Logging / saving ──────────────────────────────────────
    output_dir=OUTPUT_DIR,
    logging_steps=1,
    save_steps=50,
    save_total_limit=2,
    report_to="none",
    seed=42,
    run_name="adapt-atc-grpo-a100",
)

print("✅ GRPOConfig ready")
print(f"   precision     : bf16")
print(f"   batch         : {training_args.per_device_train_batch_size} × accum {training_args.gradient_accumulation_steps} = {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps} eff.")
print(f"   generations   : {training_args.num_generations} per prompt")
print(f"   completion len: {training_args.max_completion_length} tokens")
print(f"   KL coeff      : {training_args.beta}")
print(f"   output        : {OUTPUT_DIR}")

## Step 10 — Train!

In [ ]:
import time, os

os.makedirs(OUTPUT_DIR, exist_ok=True)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[reward_fn],
    args=training_args,
    train_dataset=dataset,
)

print("🚀 GRPO training started on A100 (bf16)!")
print("   Watch for reward_mean increasing in the log.")
print(f"   A100 estimate: ~{N_EPISODES*30//60}–{N_EPISODES*45//60} min for {N_EPISODES} episodes\n")

t0 = time.time()
trainer.train()
elapsed = time.time() - t0

print(f"\n✅ Training done in {elapsed/60:.0f} min")

## Step 11 — Save model

In [ ]:
import json

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

log_path = f"{OUTPUT_DIR}/log_history.json"
with open(log_path, "w") as f:
    json.dump(trainer.state.log_history, f, indent=2, default=str)

print(f"✅ Model saved : {OUTPUT_DIR}")
print(f"   Log         : {log_path}")

## Step 12 — Reward curve plot

In [ ]:
import matplotlib.pyplot as plt

log = trainer.state.log_history

reward_vals = []
for entry in log:
    for k in ["rewards/reward_fn", "reward", "train/reward",
              "rewards/combined_reward_fn", "rewards/reward_fn_mean"]:
        if k in entry and isinstance(entry[k], (int, float)):
            reward_vals.append(float(entry[k]))
            break

if not reward_vals:
    for entry in log:
        for k, v in entry.items():
            if "reward" in k.lower() and isinstance(v, (int, float)):
                reward_vals.append(float(v))
                break

if reward_vals:
    W  = max(5, len(reward_vals) // 15)
    ma = [sum(reward_vals[max(0,i-W):i+1]) / min(i+1, W) for i in range(len(reward_vals))]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].plot(reward_vals, alpha=0.4, color="royalblue", label="step reward")
    axes[0].plot(ma, color="darkorange", linewidth=2.5, label=f"MA-{W}")
    axes[0].set_xlabel("Training step"); axes[0].set_ylabel("Reward")
    axes[0].set_title("ADAPT ATC — Reward Curve (A100 bf16)"); axes[0].legend()
    axes[0].grid(True, alpha=0.25)

    q = max(1, len(reward_vals) // 4)
    first_q_mean = sum(reward_vals[:q]) / q
    last_q_mean  = sum(reward_vals[-q:]) / q
    axes[1].bar(["First 25%", "Last 25%"], [first_q_mean, last_q_mean],
                color=["salmon", "lightgreen"], edgecolor="black")
    for i, v in enumerate([first_q_mean, last_q_mean]):
        axes[1].text(i, v + 0.01, f"{v:.3f}", ha="center", fontweight="bold")
    axes[1].set_ylabel("Mean Reward"); axes[1].set_title("Training Improvement")
    axes[1].set_ylim(0, max(last_q_mean * 1.3, 0.2))

    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/reward_curve_a100.png", dpi=150, bbox_inches="tight")
    plt.show()

    improvement = last_q_mean - first_q_mean
    arrow = "↑" if improvement > 0.02 else ("↓" if improvement < -0.02 else "→")
    print(f"\n📊 Reward summary:")
    print(f"   First 25% mean : {first_q_mean:.4f}")
    print(f"   Last 25% mean  : {last_q_mean:.4f}")
    print(f"   Improvement    : {improvement:+.4f} {arrow}")
else:
    print("⚠️  No reward values in log_history.")
    print(f"   Available keys: {list(log[0].keys()) if log else []}")

## Step 13 — Inference demo

In [ ]:
import torch, json, re

model.eval()


def infer(role: str = "AMAN", task_id: str = "bengaluru_irrops_hard",
          max_new: int = 512, temperature: float = 0.3) -> str:
    from tasks import task_catalog
    task   = task_catalog()[task_id]
    system = AMAN_SYSTEM if role == "AMAN" else DMAN_SYSTEM

    flights_text = "\n".join(
        f"  {f.flight_id}: {f.operation.value} {f.wake_class.value} "
        f"sched={f.scheduled_minute}min [{f.earliest_minute}-{f.latest_minute}] "
        f"runways={f.allowed_runways} priority={f.priority.value}"
        for f in task.flights
    )
    user = (
        f"Task: {task_id}\nRunways: {[r.runway_id for r in task.runways]}\n"
        f"Flights:\n{flights_text}\n\nProduce the optimal {role} plan."
    )

    messages = [{"role": "system", "content": system},
                {"role": "user",   "content": user}]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(
        prompt, return_tensors="pt", truncation=True, max_length=3072
    ).to(model.device)

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new,
            temperature=temperature,
            do_sample=temperature > 0.01,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)


for task_id in ["delhi_monsoon_recovery_easy", "bengaluru_irrops_hard"]:
    print("=" * 60)
    print(f"AMAN inference — {task_id}")
    print("=" * 60)
    response = infer("AMAN", task_id)
    print(response[:600])

    clean = re.sub(r"```(?:json)?\s*|\s*```", "", response).strip()
    try:
        data  = json.loads(clean)
        slots = data.get("arrival_slots", [])
        print(f"\n✅ Valid JSON — {len(slots)} arrival slots")
        for s in slots[:5]:
            print(f"   {s.get('flight_id','?'):8s} → {s.get('runway','?'):4s} @ min {s.get('assigned_minute','?')}")
    except json.JSONDecodeError:
        print("ℹ️  Partial JSON")
    print()

## Step 14 — Multi-agent eval

In [ ]:
from multi_agent.inference import run_episode
from multi_agent.environment import MultiAgentATCEnvironment

tasks = ["delhi_monsoon_recovery_easy", "mumbai_bank_balance_medium",
         "bengaluru_irrops_hard", "hyderabad_cargo_crunch_hard"]
env   = MultiAgentATCEnvironment(seed=99)

print("Heuristic baseline (no LLM):")
composites = []
for tid in tasks:
    try:
        r = run_episode(task_id=tid, client=None, env=env, episode_id=0)
        c = r.get("composite", 0)
        composites.append(c)
        print(f"  {tid:35s}  composite={c:.3f}  "
              f"aman={r.get('aman_reward',0):.3f}  dman={r.get('dman_reward',0):.3f}")
    except Exception as e:
        print(f"  {tid}: {e}")

if composites:
    print(f"\n  Mean composite: {sum(composites)/len(composites):.3f}")

print("\n✅ For trained model eval:")
print("   python training/train_grpo.py --eval_only --model ./adapt-atc-a100-out")